1. Load the text dataset

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import urllib.request


# Download Tiny Shakespeare dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
file_name = "shakespeare.txt"


urllib.request.urlretrieve(url, file_name)

# Read the text
with open(file_name, "r", encoding="utf-8") as f:
    text = f.read()

print("Number of characters:", len(text))
print("\nSample text:")
print(text[:500])

Number of characters: 1115394

Sample text:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


2. Convert the text into character sequences

In [2]:
# Get all unique characters
characters = sorted(list(set(text)))

vocab_size = len(characters)

print("Vocabulary size:", vocab_size)
print("Characters:", characters)

# Character-to-integer mapping
char_to_idx = {char: i for i, char in enumerate(characters)}

# Integer-to-character mapping
idx_to_char = {i: char for i, char in enumerate(characters)}

# Encode the entire text
encoded_text = torch.tensor(
    [char_to_idx[c] for c in text],
    dtype=torch.long
)

print(encoded_text[:50])


Vocabulary size: 65
Characters: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56])


3. Create sequences for training

In [3]:
class TextDataset(Dataset):
    def __init__(self, encoded_text, sequence_length=100):
        self.data = encoded_text
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.data) - self.sequence_length

    def __getitem__(self, index):
        x = self.data[index:index + self.sequence_length]

        y = self.data[
            index + 1:index + self.sequence_length + 1
        ]

        return x, y

sequence_length = 100
batch_size = 64

dataset = TextDataset(
    encoded_text,
    sequence_length=sequence_length
)

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True
)

print("Number of training sequences:", len(dataset))

Number of training sequences: 1115294


4. Define the LSTM RNN model

In [4]:
class CharLSTM(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_size=128,
        hidden_size=256,
        num_layers=2
    ):
        super(CharLSTM, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Convert character IDs into embeddings
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_size
        )

        # LSTM
        self.lstm = nn.LSTM(
            input_size=embedding_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )

        # Output layer
        self.fc = nn.Linear(
            hidden_size,
            vocab_size
        )

    def forward(self, x, hidden=None):

        # x:
        # batch_size x sequence_length

        x = self.embedding(x)

        # Run LSTM
        output, hidden = self.lstm(x, hidden)

        # Predict character scores
        output = self.fc(output)

        return output, hidden


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

model = CharLSTM(
    vocab_size=vocab_size,
    embedding_size=128,
    hidden_size=256,
    num_layers=2
).to(device)

print(model)


# loss function and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.002
)



Using device: cuda
CharLSTM(
  (embedding): Embedding(65, 128)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=256, out_features=65, bias=True)
)


5. Train the LSTM model

In [5]:
epochs = 5

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for batch_number, (inputs, targets) in enumerate(dataloader):

        inputs = inputs.to(device)
        targets = targets.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs, _ = model(inputs)

        # outputs:
        # [batch_size, sequence_length, vocab_size]

        # Reshape for CrossEntropyLoss
        outputs = outputs.reshape(-1, vocab_size)

        targets = targets.reshape(-1)

        # Calculate loss
        loss = criterion(outputs, targets)

        # Backpropagation
        loss.backward()

        # Prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5
        )

        # Update parameters
        optimizer.step()

        total_loss += loss.item()


    average_loss = total_loss / (batch_number + 1)

    print(
        f"Epoch {epoch + 1}/{epochs}, "
        f"Loss: {average_loss:.4f}"
    )

Epoch 1/5, Loss: 1.2297
Epoch 2/5, Loss: 1.1221
Epoch 3/5, Loss: 1.1059
Epoch 4/5, Loss: 1.0986
Epoch 5/5, Loss: 1.0949


6. Generate new text

In [6]:
def generate_text(
    model,
    start_text,
    length=500,
    temperature=1.0
):

    model.eval()

    # Encode starting text
    input_ids = [
        char_to_idx[c]
        for c in start_text
        if c in char_to_idx
    ]

    generated_text = start_text

    hidden = None

    # Feed starting sequence into model
    input_tensor = torch.tensor(
        input_ids,
        dtype=torch.long
    ).unsqueeze(0).to(device)

    with torch.no_grad():

        output, hidden = model(
            input_tensor,
            hidden
        )

        # Generate one character at a time
        for _ in range(length):

            # Get prediction for last position
            logits = output[:, -1, :]

            # Apply temperature scaling
            logits = logits / temperature

            # Convert scores to probabilities
            probabilities = torch.softmax(
                logits,
                dim=-1
            )

            # Sample next character
            next_id = torch.multinomial(
                probabilities,
                num_samples=1
            ).item()

            next_character = idx_to_char[next_id]

            generated_text += next_character

            # Make sampled character the next input
            next_input = torch.tensor(
                [[next_id]],
                dtype=torch.long
            ).to(device)

            output, hidden = model(
                next_input,
                hidden
            )

    return generated_text



# Generate Approximately 500 new characters

7. Different temp

In [8]:

print("\nTEMPERATURE = 0.3\n")
print(
    generate_text(
        model,
        "ROMEO:",
        length=300,
        temperature=0.3
    )
)

print("\nTEMPERATURE = 0.8\n")
print(
    generate_text(
        model,
        "ROMEO:",
        length=300,
        temperature=0.8)
)


print("\nTEMPERATURE = 1.5\n")
print(
    generate_text(
        model,
        "ROMEO:",
        length=300,
        temperature=1.5
    )
)


TEMPERATURE = 0.3

ROMEO:
I will make the same through the sign of death.

CLIFFORD:
No man live: I have not so fair and stay,
That thou shalt be the sea, the tortured slave
To whom I were a forenature for the ground.

KING RICHARD III:
So shall the state of heaven sleep in love;
And then we shall not have strong afford his

TEMPERATURE = 0.8

ROMEO:
Why, so: do you be a consul, in twenty Juliet?

PETRUCHIO:
A shut up that day! let's all request in them.

KING HENRY VI:
If this is the storm at this senator and words;
Yet then dissemblers and a foot in spark:
And we will demand, my liege, my lord, nor woman-life
That I in this mind come from thi

TEMPERATURE = 1.5

ROMEO:
You.

AUTOLYCUS:
They speakable voice,
And mightiting,--
Up take not him there. Let his face I hope:
And, you shall live:
Gentle feeling, but welcome! Look, let's part
There it learns; his noble hatch of venit,
honest lamp, get noble than any thing, Isabel.

ELBOW:
Come, kind enrol,' piteous,
speck
